# Welcome to Colab!

In [ ]:
!pip install -q langchain langchain-community langchain-groq
!pip install -q langchain-huggingface sentence-transformers
!pip install -q faiss-cpu pypdf gradio


In [ ]:
import os

from google.colab import files

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

from langchain_groq import ChatGroq

In [ ]:
uploaded = files.upload()

Saving Local_News_Summarization_Practice.pdf to Local_News_Summarization_Practice (2).pdf


In [ ]:
import os

groq_api_key = "******************************"

os.environ["GROQ_API_KEY"] = groq_api_key

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

response = llm.invoke("What is local news summarization?")

print(response.content)

**Local news summarization** is the process of automatically creating a short, coherent version of a news story that focuses on events, issues, or topics relevant to a specific geographic area (city, region, neighborhood, etc.). In other words, it’s a way of distilling the most important information from a locally‑focused article—or a collection of local articles—into a concise summary that a reader can grasp quickly.

---

## Why It Matters

| Goal | Benefit |
|------|----------|
| **Speed** | Readers can get the gist of a story in seconds instead of minutes. |
| **Relevance** | Highlights what matters most to a particular community (e.g., a road closure, a school board decision, a local crime). |
| **Accessibility** | Helps people with limited time, low bandwidth, or reading difficulties. |
| **Information Overload** | Filters out noise when many local outlets publish overlapping reports. |
| **Personalization** | Powers recommendation engines that serve each user the most pertinent 

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Local_News_Summarization_Practice.pdf to Local_News_Summarization_Practice (3).pdf


In [ ]:
groq_api_key = "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"

In [ ]:
os.environ["GROQ_API_KEY"] = groq_api_key

In [ ]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [ ]:
response = llm.invoke("What is Artificial Intelligence?")

print(response.content)

**Artificial Intelligence (AI)** is a branch of computer science that focuses on creating systems capable of performing tasks that normally require human intelligence. These tasks include learning, reasoning, problem‑solving, perception, language understanding, and decision‑making.

### Core Concepts

| Concept | What It Means | Example |
|---------|---------------|---------|
| **Machine Learning (ML)** | Algorithms that improve automatically through experience (data). | Spam‑filter that gets better at spotting junk mail as it processes more messages. |
| **Deep Learning** | A subset of ML using multi‑layered neural networks to model complex patterns. | Image‑recognition systems that can identify objects in photos. |
| **Natural Language Processing (NLP)** | Techniques for understanding and generating human language. | Chatbots, translation services, sentiment analysis. |
| **Computer Vision** | Enabling machines to interpret visual information. | Self‑driving cars detecting pedestrian

In [ ]:
!pip install -q pypdf langchain-community

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
print(PyPDFLoader)

<class 'langchain_community.document_loaders.pdf.PyPDFLoader'>


In [ ]:
pdf_path = "Local_News_Summarization_Practice.pdf"

loader = PyPDFLoader(pdf_path)

documents = loader.load()

print("PDF loaded successfully!")
print("Number of pages:", len(documents))

PDF loaded successfully!
Number of pages: 2


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

Number of chunks: 6


In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS vector database created successfully!")

FAISS vector database created successfully!


In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [ ]:
question = "What is the main topic of this document?"

results = retriever.invoke(question)

for i, doc in enumerate(results):
    print("\n--- Result", i + 1, "---")
    print(doc.page_content[:500])


--- Result 1 ---
people or organizations involved, important facts, effect on residents, and any action or advice mentioned in the
article.
12. Questions for Testing Your RAG Agent
1. What happened in the road development news?
2. Which city is covered in these articles?
3. What problems were caused by heavy rainfall?
4. What topics were presented at the college technology event?
5. What was the purpose of the health-awareness programme?
6. What is being reviewed in public transportation?
7. What activities were

--- Result 2 ---
10. Example Summaries
Road Development: Coimbatore is planning phased road improvements to repair damaged roads and improve
traffic safety.
Rainfall: Heavy rain caused temporary waterlogging in some areas, leading to slower traffic and drainage work.
Technology Event: College students showcased AI, robotics, and software projects at a local technology
event.
Health Awareness: A community programme promoted preventive healthcare, cleanliness, healthy habits, a

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Answer the question using only the information provided
in the context.

If the answer is not available in the context, say:
"I could not find this information in the PDF."

Context:
{context}

Question:
{question}

Answer:
""")

In [ ]:
def rag_answer(question):
    docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    final_prompt = prompt.format(
        context=context,
        question=question
    )

    response = llm.invoke(final_prompt)

    return response.content

In [ ]:
answer = rag_answer(
    "What is the main topic of the PDF?"
)

print(answer)

The PDF is a fictional practice dataset created for learning and testing a local‑news summarization agent.


In [ ]:
from langchain_core.tools import tool

@tool
def pdf_rag_tool(question: str) -> str:
    """
    Search the uploaded PDF and answer questions
    using information from the PDF.
    """

    return rag_answer(question)

In [ ]:
@tool
def budget_tool(amount: float, percentage: float) -> str:
    """
    Calculate a percentage of an amount.
    """

    result = amount * percentage / 100

    return f"{percentage}% of ₹{amount} is ₹{result:.2f}"

In [ ]:
print(
    budget_tool.invoke({
        "amount": 10000,
        "percentage": 20
    })
)

20.0% of ₹10000.0 is ₹2000.00


In [ ]:
from langchain.agents import create_agent

tools = [
    pdf_rag_tool,
    budget_tool
]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
You are a helpful PDF assistant.

You can:
1. Answer questions using the uploaded PDF.
2. Calculate budgets when required.

Use the PDF RAG tool for questions about the PDF.
Use the budget tool for calculations.
"""
)

In [ ]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the main topic of the PDF?"
        }
    ]
})

print(result["messages"][-1].content)

The PDF is a fictional practice dataset created for learning and testing a local‑news summarization agent.


In [ ]:
chat_history = []

In [ ]:
def ask_agent(question):

    chat_history.append({
        "role": "user",
        "content": question
    })

    result = agent.invoke({
        "messages": chat_history
    })

    answer = result["messages"][-1].content

    chat_history.append({
        "role": "assistant",
        "content": answer
    })

    return answer

In [ ]:
print(ask_agent("My budget is ₹20,000."))

Got it! Your budget is ₹20,000. How can I help you with it? For example, would you like to:

- Allocate percentages to different expense categories?  
- Compare it against figures in the PDF you uploaded?  
- Run any specific calculations?

Just let me know what you need!


In [ ]:
print(ask_agent("What is 15% of it?"))

₹3,000 is 15 % of your ₹20,000 budget. Let me know if you’d like to break down the remaining amount or do any other calculations!


In [ ]:
import gradio as gr

In [ ]:
def chat(message, history):

    answer = ask_agent(message)

    return answer

In [ ]:
demo = gr.ChatInterface(
    fn=chat,
    title="PDF RAG Agent",
    description="Ask questions about your PDF or use the budget calculator."
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://43a22d19870f2579ca.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
